Question 5

In [2]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

In [3]:
TARGET_COL = "price"
DROP_COLS = ["id", "date", "zipcode", "Unnamed: 0"]

def load_prepare(path: str):
    df = pd.read_csv(path)

    y = df[TARGET_COL].astype(float).to_numpy() / 1000.0
    X = df.drop(columns=[TARGET_COL] + [c for c in DROP_COLS if c in df.columns], errors="ignore")
    X = X.select_dtypes(include = [np.number])
    return X , y

def add_intercept(X: np.ndarray):
    return np.column_stack([np.ones(X.shape[0]), X])

def gradient_descent(X: np.ndarray, y: np.ndarray, alpha: float, num_iters: int):
    #Uses cost J(theta) = (1/(2m)) * ||X theta - y||^2.
    #Gradient = (1/m) * X^T (X theta - y).
    m, d = X.shape
    X_aug = add_intercept(X)
    theta = np.zeros(d + 1)

    for _ in range(num_iters):
        preds = X_aug @ theta
        grad = (1 / m) * (X_aug.T @ (preds - y))
        theta = theta - alpha * grad

    return theta

def predict(X: np.ndarray, theta: np.ndarray):
    return add_intercept(X) @ theta

def evaluate(y_true, y_pred):
    return mean_squared_error(y_true, y_pred), r2_score(y_true, y_pred)


In [4]:
train_path = "train.csv"
test_path  = "test.csv"

X_train_df, y_train = load_prepare(train_path)
X_test_df,  y_test  = load_prepare(test_path)

X_train_df, X_test_df = X_train_df.align(X_test_df, join = "inner", axis = 1)
feature_names = X_train_df.columns.tolist()
imputer = SimpleImputer(strategy = "median")
scaler = StandardScaler()

X_train = scaler.fit_transform(imputer.fit_transform(X_train_df))
X_test  = scaler.transform(imputer.transform(X_test_df))

alphas = [0.01, 0.1, 0.5]
iters_list = [10, 50, 100]

rows = []
thetas = {}

for a in alphas:
    for iters in iters_list:
        theta = gradient_descent(X_train, y_train, a, iters)
        thetas[(a, iters)] = theta

        pred_tr = predict(X_train , theta)
        pred_te = predict(X_test, theta)

        tr_mse, tr_r2 = evaluate(y_train, pred_tr)
        te_mse, te_r2 = evaluate(y_test, pred_te)

        rows.append([a, iters, tr_mse, tr_r2, te_mse, te_r2])

results = pd.DataFrame(rows, columns=["alpha", "iters", "Train MSE", "Train R2", "Test MSE", "Test R2"])
print(results)

#theta values
print("\nParameter order:" , ["intercept"] + feature_names)
for a in alphas:
    for iters in iters_list:
        print(f"\nalpha={a}, iters={iters}")
        print(thetas[(a, iters)])

   alpha  iters     Train MSE      Train R2      Test MSE       Test R2
0   0.01     10  2.947987e+05 -1.560413e+00  3.505251e+05 -1.102390e+00
1   0.01     50  1.382959e+05 -2.011404e-01  1.703767e+05 -2.189040e-02
2   0.01    100  7.011899e+04  3.909961e-01  9.748624e+04  4.152940e-01
3   0.10     10  6.649932e+04  4.224340e-01  9.355929e+04  4.388472e-01
4   0.10     50  3.157898e+04  7.257273e-01  5.801232e+04  6.520519e-01
5   0.10    100  3.149769e+04  7.264333e-01  5.772519e+04  6.537741e-01
6   0.50     10  6.118299e+08 -5.312921e+03  6.850231e+08 -4.107652e+03
7   0.50     50  1.649496e+25 -1.432635e+20  1.842083e+25 -1.104850e+20
8   0.50    100  5.698752e+45 -4.949532e+40  6.364111e+45 -3.817086e+40

Parameter order: ['intercept', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'lat', 'long', 'sqft_living15', 'sqft_lot15']

alpha=0.01, iters=10
[ 4.97609866e+0

With gradient descent, the learning rate α controls both how fast the model improves and whether it stays stable. For the small learning rate α = 0.01, the algorithm improves slowly. For α = 0.1, convergence is much faster-- by 50 iterations the metrics are already close to their best values (Train MSE ~ 3.16 * 10^4, Train R^2 ~ 0.726, Test MSE ~ 5.8 * 10^4, and Test R^2 ~ 0.652), and going to 100 iterations only changes the results slightly. However, α = 0.5 is too large and the algorithm diverges, with MSE raising up to extremely large values and R^2 becoming very negative, which means the updates overestimated the minimum and theta got huge.